# Neural Network Builder for Image Classification Tasks with CIFAR10 and Tensorflow

## 1. Introduction
### 1.1 Summary
This notebooks contains a step by step example of building a convolutional neural network model with Tensorflow.
<br>
Based on the <a href="https://www.cs.toronto.edu/~kriz/cifar.html" target="_blank">CIFAR10 Dataset</a>,
this model is suitable for performing **classification tasks**, recognizing 10 different entities:
<br>
airplanes, automobiles, birds, cats, deer, dogs, frogs, horses, ships and trucks.

### 1.2 Requirements
The requiements to run this notebook can be installed via requirements.txt, as demonstrated in <a href="https://video/url/goes/here" trarget="_blank">this tutorial</a>.<br>
Or alternativley, you can do it directly from this notebook by running the following cell:

In [3]:
# # for windows/linux
# # !pip install tensorflow
# # for mac
# !pip install tensorflow-macos
# !pip install tensorflow-metal
# !pip install numpy
# !pip install matplotlib
# !pip install pillow

### 1.3 Imports

In [4]:
# M2 Pro Mac - TensorFlow Configuration
import os

os.environ['TF_CPP_MINIMUM_LOG_LEVEL'] = '2'

import warnings

warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt

# TensorFlow imports - M2 compatible with fallback
try:
    import tensorflow as tf

    # Safely check TensorFlow version
    try:
        tf_version = tf.__version__
    except AttributeError:
        tf_version = "unknown (broken installation)"

    print(f"✓ TensorFlow {tf_version} imported")

    # Try importing keras from tensorflow
    try:
        from tensorflow import keras

        print("✓ Keras imported from tensorflow")
    except (ImportError, AttributeError):
        # Fallback: import keras directly
        try:
            import keras

            print("✓ Keras imported directly (standalone)")
        except ImportError:
            raise ImportError("Neither tensorflow.keras nor standalone keras could be imported")

    from tensorflow.keras import Sequential, layers, datasets, callbacks, utils
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    from tensorflow.keras.optimizers.legacy import Adam

except (ImportError, AttributeError) as e:
    print(f"❌ TensorFlow import error: {e}")
    print("\nTensorFlow may be broken. Try fixing it with:")
    print("  pip uninstall tensorflow tensorflow-macos tensorflow-metal -y")
    print("  pip cache purge")
    print("  pip install tensorflow-macos tensorflow-metal")
    raise


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/anaconda3/envs/NeuroLens/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/anaconda3/envs/NeuroLens/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/anaconda3/envs/NeuroLens/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/envs/NeuroLens/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instan

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

### GPU Configuration for M2 Mac

In [ ]:
# M2 GPU Optimization (only if TensorFlow imported successfully)
try:
    if not hasattr(tf, 'config'):
        print("⚠ TensorFlow config not available - installation may be broken")
        print("Trying to fix TensorFlow...")
        raise AttributeError("TensorFlow config missing")

    gpu_devices = tf.config.list_physical_devices('GPU')
    if gpu_devices:
        try:
            for gpu in gpu_devices:
                tf.config.experimental.set_memory_growth(gpu, True)
            print("✓ GPU memory growth enabled")
        except RuntimeError as e:
            print(f"⚠ GPU config warning: {e}")

    # Thread optimization for M2
    tf.config.threading.set_intra_op_parallelism_threads(8)
    tf.config.threading.set_inter_op_parallelism_threads(8)
    print("✓ M2 thread optimization applied")
    print("✓ Ready to train on M2 GPU (Metal)")
except (Exception, AttributeError) as e:
    print(f"⚠ GPU configuration skipped: {e}")
    print("\nTensorFlow installation may be broken.")
    print("Try reinstalling with:")
    print("  pip uninstall tensorflow tensorflow-macos tensorflow-metal -y")
    print("  pip cache purge")
    print("  pip install tensorflow-macos tensorflow-metal")
    print("\nYou can still try to train on CPU, but it will be slower")

## 2. Load Dataset
### 2.1 Fetch Data
in the following cell we will load the CIFAR10 dataset, and check the shape and total number of dataset samples.

In [ ]:
print("Loading CIFAR-10 dataset...")
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
print(f"✓ Dataset loaded successfully!")

print("{} train samples and {} test samples\n".format(len(x_train), len(x_test)))
print("train samples shape:", x_train.shape)
print("train labels shape:", y_train.shape)

### 2.2 Data Overview

#### 2.2.1 Train Versus Test
we see that CIFAR10 contains:
- ```50,000``` train samples
- ```10,000``` test samples

#### 2.2.2 Sample Structure
all samples are:
- ```32px``` wide
- ```32px``` high
- ```3``` colour channels deep (in other words, ```RGB``` images)

#### 2.2.3 Label Structure
all labels contain a single value, that represents the entity that appears in the sample.
<br>
**for example:** if the sample is a photo of a ```cat``` - then the label will be ```3```.

### 2.3 Translate Class Labels to Class Names
the following dictionary maps the numeric labels of CIFAR10 to their corresponding class names.<br>
once we run it, we won't have to worry about deciphering the meaning behind the numbers - we can easily convert them to their name representation.

In [ ]:
class_names = {
    0: 'airplane',
    1: 'automobile',
    2: 'bird',
    3: 'cat',
    4: 'deer',
    5: 'dog',
    6: 'frog',
    7: 'horse',
    8: 'ship',
    9: 'truck',
}

### 2.4 Visualize Dataset

Let's plot 9 random samples from the dataset along with their labels to verify our mapping is correct.
<br>
each time this cell runs, a different batch of samples will be plotted.

In [ ]:
# select a random set of 9 images
idx = np.random.randint(len(x_train) - 9)

plt.figure(figsize=(10, 10))
for i in range(9):
    # plot each sample
    plt.subplot(3, 3, i + 1)
    plt.imshow(x_train[i + idx])
    plt.xlabel(class_names[(y_train[i + idx][0])])

# display results
plt.show()

## 3. Data Reduction

before loading the data into a neural network, we will implement some data reduction tenchiques to ensure maximum efficiency.

### 3.1 Samples Reduction

### 3.1.1 Current Sample Values
Each pixel in each image stores colour intensity values in the range of ```0-255```, where:
- 0 represents the minimum colour intencity.
- 255 represents the maximum clour intencity.

RGB representation:
- in RGB, **pure red** is denoted with (255, 0, 0) with maximum intensity on the red channel.
- in RGB, **pure green** is denoted with (0, 255, 0) with maximum intensity on the green channel.
- in RGB, **pure blue** is denoted with (0, 0, 255) with maximum intensity on the blue channel.
- in RGB, **pure yellow** is denoted with (255, 255, 0) with maximum intensity on the red and green channels.

let's quickly print an example of some pixel values from the first train sample.

In [ ]:
# [first sample], [first colour channel], [first row of pixels]
print(x_train[0][0][0])

### 3.1.2 Sample Normalization

The problem is - the values we just printed span across a very large range!
<br>
We can, in fact, reduce it by mapping each value in the range of ```0-255``` to an equivalent value in the range of ```0-1```.
<br>
this technique is called **Normalization** where the relationship between the values remains the same, but they are reduced to a smaller scale.
<br>
We'll simply divide each pixel value in our samples by 255, like so:

In [ ]:
x_train = x_train / 255.
x_test = x_test / 255.

### 3.1.3 Updated Sample Values

When we print the exact same example, we now get the same values but at a different scale.
<br>
We can actually verify that our images did not change their appearance, by **re-running the plotting cell** from above.
<br>
You will see that our reduction technique did not result in any loss of information.

In [ ]:
print(x_train[0][0][0])

### 3.2 Labels Reduction

#### 3.2.1 Current Label Values
currently, our label stores an integer value that represents a certain class

In [ ]:
print("class {} represents a {}".format(y_train[0][0], class_names[y_train[0][0]]))

### 3.2.2 One Hot Encoding

The problem is - once again we are dealing with a **relativley large search space** of values in the range of ```0-9```.
<br>
However, we cannot normaize these values because they represent distinct categories (in other words - they are discrete and therefore we cannot treat them like continous values). In these cases we use a different technique called **One Hot Encoding**.
<br>
Where we convert decimal values to their binary representation. Such that:
- the class of ```0``` is One-Hot-Encoded into ```1000000000```
- the class of ```1``` is One-Hot-Encoded into ```0100000000```
- the class of ```2``` is One-Hot-Encoded into ```0010000000```
- the class of ```9``` is One-Hot-Encoded into ```0000000001```

Let's quickly implement it in the next cell

In [ ]:
from tensorflow.keras.utils import to_categorical

y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

### 3.2.3 Updated Label Values

When we print the exact same label, we now get the same values but they are represented differently.

In [ ]:
print("class {} represents a {}".format(y_train[0], class_names[np.argmax(y_train[0])]))

## 4. Create Baseline Neural Netwok

Now it's time for a fun experiment! We will pick some arbitrary parameters for what we call a **Baseline Model**. This is the initial neural network that we begin our exploration process with. I highly encourage you to change some of the parameters and find how they affect the resuts.

## 4.1 Exploration Rules
you can remove, add or change the layers and their parameters, as long as you follow the following rules
- do not change the input shape, it must remain: ```input_shape=(32, 32, 3)```
- do not change the output units of the last Dense layer, it must remain: ```layers.Dense(10)```
- do not remove the Softmax layer, it must be last: ```layers.Softmax()```
- do not remove the Flatten layer, it must remain in between the Convolutional/MaxPooling layers and the Dense ones: ```layers.Flatten()```
- do not change the loss, it must remain ```loss = 'categorical_crossentropy'```
- do not change the evaluation metrics, as they are associated with the lost, they must remain at: ```metrics = ['accuracy']```

## 4.2 Create Model

In [ ]:
from tensorflow.keras import Sequential, layers


def build_model():
    """Build an improved CNN model with batch normalization and regularization."""
    neural_model = Sequential()

    # First Conv Block
    neural_model.add(layers.Conv2D(64, 3, activation='relu', padding='same', input_shape=(32, 32, 3)))
    neural_model.add(layers.BatchNormalization())
    neural_model.add(layers.Conv2D(64, 3, activation='relu', padding='same'))
    neural_model.add(layers.BatchNormalization())
    neural_model.add(layers.MaxPooling2D(pool_size=(2, 2)))
    neural_model.add(layers.Dropout(0.25))

    # Second Conv Block
    neural_model.add(layers.Conv2D(128, 3, activation='relu', padding='same'))
    neural_model.add(layers.BatchNormalization())
    neural_model.add(layers.Conv2D(128, 3, activation='relu', padding='same'))
    neural_model.add(layers.BatchNormalization())
    neural_model.add(layers.MaxPooling2D(pool_size=(2, 2)))
    neural_model.add(layers.Dropout(0.25))

    # Third Conv Block
    neural_model.add(layers.Conv2D(256, 3, activation='relu', padding='same'))
    neural_model.add(layers.BatchNormalization())
    neural_model.add(layers.MaxPooling2D(pool_size=(2, 2)))
    neural_model.add(layers.Dropout(0.25))

    # Flatten and Dense Layers
    neural_model.add(layers.Flatten())
    neural_model.add(layers.Dense(512, activation='relu'))
    neural_model.add(layers.BatchNormalization())
    neural_model.add(layers.Dropout(0.5))
    neural_model.add(layers.Dense(256, activation='relu'))
    neural_model.add(layers.BatchNormalization())
    neural_model.add(layers.Dropout(0.3))
    neural_model.add(layers.Dense(10))
    neural_model.add(layers.Softmax())

    # Use legacy Adam optimizer for M2 Mac (faster performance)
    neural_model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy'])
    return neural_model


# create model
model = build_model()

# see model details
model.summary()

## 5. Train Model

Once our data and model are ready, we can proceed with a process called training, where the model reviews and learns the data time and time again.
<br>
The process of repetition, allows the model to recognize patterns within the data as it observes it from different angles through different filters.

### 5.1 Beyond the Scope: Validation
Usually, to train a Baseline Model such as ours, we must perform a process called "validation". However, it's beyond the scope of this notebook, I will cover it in future tutorials.
<br>
If you'd still like to perform validation, this code will collect **both train and validation metrics** rather than just the train ones, as we do in the next cell:
- train metric: ```history.history['accuracy']```
- train metric: ```history.history['loss']```
- validation metric: ```history.history['val_accuracy']```
- validation metric: ```history.history['val_loss']```

```
# reserve samples for validation
valset_size = len(x_train) // 5
x_val = x_train[-valset_size:]
y_val = y_train[-valset_size:]
x_partial_train = x_train[:-valset_size]
y_partial_train = y_train[:-valset_size]

history = model.fit(
    x_train,
    y_train,
    epochs=2,
    validation_data=(x_val, y_val),
)
```

### 5.2 Train Rules
- **Epochs**: Each data sample passes through each model layer exactly ```number_of_epochs``` times. <br>Where epoch represents one model iteration over the data, in our case - we have ```10``` of them. <br>Please feel free to modify this value, but keep in mind - the more epochs the longer your model will train!
- **Save Model**: Once the training ends, we save our model as ```baseline.keras``` so we can load it and re-use it in the future without training.
- **History**: the history object stores a dictionary that tracks the ```accuracy``` and ```loss``` metrics for each epoch.
- **Loss**: loss must reduce over time, otherwise something is wrong!
- **Accuracy**: accuracy must improve over time, otherwise something is wrong!

### 5.3 Data Augmentation (Optional but Recommended)

Data augmentation improves model generalization by applying random transformations to training images:

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Create data augmentation generator
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    fill_mode='nearest'
)

# Fit the generator on training data
train_datagen.fit(x_train)

### 5.4 Setup Callbacks for Training

Early stopping and model checkpoint callbacks:

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Create callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    '../assets/baseline.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

callbacks_list = [early_stop, model_checkpoint]

### 5.5 Perform Training with Validation

Now let's train the improved model with data augmentation and validation:

In [ ]:
# Reserve samples for validation
valset_size = len(x_train) // 5
x_val = x_train[-valset_size:]
y_val = y_train[-valset_size:]
x_partial_train = x_train[:-valset_size]
y_partial_train = y_train[:-valset_size]

# Train with data augmentation
import time

start_time = time.time()

history = model.fit(
    train_datagen.flow(x_partial_train, y_partial_train, batch_size=128),
    epochs=50,
    validation_data=(x_val, y_val),
    callbacks=callbacks_list,
    verbose=1
)

training_time = time.time() - start_time
print(f"Training Time: {training_time:.2f} seconds ({training_time / 60:.2f} minutes)")

print("\nTraining completed!")
print(f"Initial train accuracy: {history.history['accuracy'][0]:.4f}")
print(f"Final train accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")

### 5.6 Plot Training Results

Let's visualize the training progress with both training and validation metrics:

In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(14, 5))

# Plot Accuracy
axis[0].plot(history.history["accuracy"], label='Train Accuracy', linewidth=2)
axis[0].plot(history.history["val_accuracy"], label='Validation Accuracy', linewidth=2)
axis[0].set_title("Model Accuracy Over Epochs", fontsize=12, fontweight='bold')
axis[0].set_xlabel("Epoch")
axis[0].set_ylabel("Accuracy")
axis[0].legend()
axis[0].grid(True, alpha=0.3)

# Plot Loss
axis[1].plot(history.history["loss"], label='Train Loss', linewidth=2)
axis[1].plot(history.history["val_loss"], label='Validation Loss', linewidth=2)
axis[1].set_title("Model Loss Over Epochs", fontsize=12, fontweight='bold')
axis[1].set_xlabel("Epoch")
axis[1].set_ylabel("Loss")
axis[1].legend()
axis[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print("\nTraining Summary:")
print(
    f"Best validation accuracy: {max(history.history['val_accuracy']):.4f} at epoch {np.argmax(history.history['val_accuracy']) + 1}")
print(f"Best validation loss: {min(history.history['val_loss']):.4f}")
print(f"Training completed with {len(history.history['accuracy'])} epochs")

## 6. Test Model

### 6.1 Perform Testing

The final stage before releasing our model - is testing it! We will use the ```x_test``` and ```y_test``` data which our model has never seen before - and check how well it classifies it.
<br>
The **test accuracy score** is given by how many examples our model **correctly classified** in total. Anything above 75% is excellent!

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f'\nTest Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc * 100:.2f}%')

# Get predictions
predictions = model.predict(x_test)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)

# Calculate per-class accuracy
from sklearn.metrics import confusion_matrix, classification_report

print("\n" + "=" * 60)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(true_classes, predicted_classes, target_names=list(class_names.values())))

# Confusion Matrix
cm = confusion_matrix(true_classes, predicted_classes)
print("\nConfusion Matrix:")
print(cm)

### 6.2 Verify Testing Results

Let's plot sample predictions to visually verify the model's performance:

#### 6.2.1 Plot Correct Predictions

In [ ]:
# Find correct predictions
correct_mask = predicted_classes == true_classes
correct_indices = np.where(correct_mask)[0][:9]

plt.figure(figsize=(12, 8))
plt.suptitle('Correct Predictions', fontsize=14, fontweight='bold')
for i, idx in enumerate(correct_indices):
    plt.subplot(3, 3, i + 1)
    plt.imshow(x_test[idx])
    pred_class = class_names[predicted_classes[idx]]
    confidence = predictions[idx][predicted_classes[idx]] * 100
    plt.xlabel(f'✓ {pred_class}\n({confidence:.1f}%)', color='green', fontweight='bold')
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()

#### 6.2.2 Plot Incorrect Predictions

In [ ]:
# Find incorrect predictions
incorrect_mask = predicted_classes != true_classes
incorrect_indices = np.where(incorrect_mask)[0][:9]

if len(incorrect_indices) > 0:
    plt.figure(figsize=(12, 8))
    plt.suptitle('Incorrect Predictions (Misclassifications)', fontsize=14, fontweight='bold')
    for i, idx in enumerate(incorrect_indices):
        plt.subplot(3, 3, i + 1)
        plt.imshow(x_test[idx])
        pred_class = class_names[predicted_classes[idx]]
        true_class = class_names[true_classes[idx]]
        confidence = predictions[idx][predicted_classes[idx]] * 100
        plt.xlabel(f'Pred: {pred_class}\nTrue: {true_class}\n({confidence:.1f}%)',
                   color='red', fontweight='bold', fontsize=9)
        plt.xticks([])
        plt.yticks([])
    plt.tight_layout()
    plt.show()
else:
    print("No incorrect predictions! Perfect accuracy!")